# Verification MCP Tools

Build and test the individual verification components: quality scorer, citation validator, and fact checker.

## Learning Objectives
- Implement rule-based citation validation
- Build LLM-based quality scoring
- Create fact-checking via cross-reference
- Understand how verification functions serve as MCP tools in the harness

In [ ]:
import os
import sys
import httpx
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
env_path = os.path.join('..', '.env')
load_dotenv(env_path, override=True)

_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")
_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

from agents.orchestrator.layers.verification import (
    quality_score,
    validate_citations,
    fact_check,
    run_verification,
)

print("Verification functions loaded")

## 1. Citation Validator

This is a **rule-based** check that verifies `[Source N]` references in the draft actually correspond to real sources.

In [ ]:
# Test with a well-cited draft
good_draft = """
Vector databases are essential for RAG applications [Source 1]. They enable efficient 
similarity search over high-dimensional embeddings [Source 2]. PostgreSQL with pgvector 
provides enterprise-grade vector storage [Source 1] with ACID compliance.
"""

sources = [
    {"document_name": "vector_db_guide.pdf", "content": "Vector databases store embeddings..."},
    {"document_name": "pgvector_docs.pdf", "content": "pgvector extends PostgreSQL..."},
]

result = validate_citations(good_draft, sources)
print("Good draft citation check:")
print(f"  Valid citations: {result['valid_citations']}")
print(f"  Invalid citations: {result['invalid_citations']}")
print(f"  Coverage: {result['citation_coverage']}")
print(f"  Passed: {result['passed']}")

In [ ]:
# Test with a poorly-cited draft
bad_draft = """
Vector databases are the best technology ever created [Source 5]. They solve all 
problems in AI [Source 99]. No other approach works as well.
"""

result = validate_citations(bad_draft, sources)
print("Bad draft citation check:")
print(f"  Valid citations: {result['valid_citations']}")
print(f"  Invalid citations: {result['invalid_citations']}")
print(f"  Has citations: {result['has_citations']}")
print(f"  Passed: {result['passed']}")

## 2. Quality Scorer

Uses an LLM to evaluate completeness, accuracy, clarity, and structure.

In [ ]:
query = "What are the benefits of vector databases for RAG?"

draft = """
# Benefits of Vector Databases for RAG

## Executive Summary
Vector databases provide efficient similarity search capabilities that are essential 
for Retrieval-Augmented Generation (RAG) applications [Source 1].

## Key Benefits
1. **Semantic Search**: Find contextually relevant documents beyond keyword matching [Source 1]
2. **Scalability**: Handle millions of embeddings with sub-second query times [Source 2]
3. **Integration**: pgvector integrates directly with PostgreSQL [Source 2]

## Conclusion
Vector databases are a critical component of production RAG systems.
"""

scores = quality_score(draft, query)
print("Quality scores:")
for key in ['completeness', 'accuracy', 'clarity', 'structure', 'overall']:
    print(f"  {key}: {scores.get(key, 'N/A')}")
print(f"  Feedback: {scores.get('feedback', 'N/A')}")
print(f"  Tokens used: {scores.get('tokens_used', 0)}")

## 3. Fact Checker

Cross-references claims in the draft against source documents.

In [ ]:
context = [
    {"content": "Vector databases enable semantic similarity search using embeddings. They outperform keyword search for finding contextually relevant documents."},
    {"content": "pgvector is a PostgreSQL extension that adds vector similarity search. It supports HNSW and IVFFlat indexing for fast retrieval."},
]

fact_result = fact_check(draft, context)
print("Fact check results:")
print(f"  Supported claims: {fact_result.get('supported_claims', 'N/A')}")
print(f"  Unsupported claims: {fact_result.get('unsupported_claims', 'N/A')}")
print(f"  Hallucinations: {fact_result.get('hallucinations', [])}")
print(f"  Passed: {fact_result.get('passed', 'N/A')}")

## 4. Aggregated Verification

The `run_verification` function combines all checks into a single result.

In [ ]:
full_result = run_verification(draft, query, context, iteration=1)

print(f"Aggregated Verification:")
print(f"  Quality score: {full_result['quality_score']}/10")
print(f"  Passed: {full_result['passed']}")
print(f"  Improvements:")
for imp in full_result.get('improvements', []):
    print(f"    - {imp}")
print(f"  Total tokens: {full_result.get('tokens_used', 0)}")

## Summary

Verification components:
- **Citation Validator**: Rule-based, fast, catches reference errors
- **Quality Scorer**: LLM-based, multi-dimensional assessment
- **Fact Checker**: LLM cross-reference against source documents
- **Aggregation**: Combined result drives iteration decisions

These verification tools form the **Verify** phase of the AGENTS.md harness inner loop.